# Translation Pipeline — `kanker_nl_breakout`

**Source**: `kanker_nl_master_table_breakout_v3.csv`  
**Output**: `kanker_nl_master_table_breakout_v3_translated.csv`  
**Trustworthiness** (from Makhotso's matrix): `4.9`  
**Volume**: 73 rows · ~94k Dutch chars · ~7 min

### How to run

1. Upload the source CSV to the same folder as this notebook (or change the path below)
2. Run all cells top to bottom
3. The output CSV will appear next to the source, with `_en` columns added

Each translated column is added as a sibling next to the Dutch original. Empty cells, URLs, hashes and dates are skipped automatically.

## 1. Install dependencies

In [1]:
pip install --quiet deep-translator pandas


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Shared translation library

*(Inlined so the notebook is self-contained — same code as `translate_lib.py`.)*

In [2]:
"""
Shared translation library for CSN Team 23 — Dutch → English.

Design choices (from facilitator session + earlier IKNL pipeline):
- deep-translator (Google backend), free tier
- Sentence-aware chunking at 4500 chars (under Google's 5000-char cap)
- 0.5s polite delay between API calls
- Skip cells that are empty, URLs, hashes, dates, or pure numbers
- Cache translations on content_hash to avoid re-translating identical Dutch
- Add a `_en` column next to each translated Dutch column, preserving source structure
"""
import re
import time
import hashlib
from typing import List, Optional
import pandas as pd
from deep_translator import GoogleTranslator

MAX_CHUNK = 4500          # under Google's 5000-char limit
DELAY_SEC = 0.5           # polite delay
RETRY = 3                 # retries per chunk on transient failure

# In-memory cache: maps SHA-1 of Dutch text → English translation
_cache: dict = {}


def _is_translatable(text) -> bool:
    """Return True if the cell holds Dutch text worth translating."""
    if pd.isna(text):
        return False
    s = str(text).strip()
    if not s:
        return False
    # Skip URLs, hashes, ISO dates, pure numbers
    if s.startswith(("http://", "https://", "www.")):
        return False
    if re.fullmatch(r"[0-9a-f]{32,128}", s):  # content hash
        return False
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}.*", s):  # ISO date
        return False
    if re.fullmatch(r"[\d\s.,+\-%]+", s):  # numbers
        return False
    if len(s) < 3:
        return False
    return True


def _split_into_sentences(text: str) -> List[str]:
    """Split on sentence boundaries while preserving punctuation."""
    # Split on . ! ? followed by whitespace
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p for p in parts if p]


def _chunk_text(text: str, max_chars: int = MAX_CHUNK) -> List[str]:
    """Group sentences into chunks ≤ max_chars."""
    if len(text) <= max_chars:
        return [text]
    sentences = _split_into_sentences(text)
    chunks: List[str] = []
    buf = ""
    for s in sentences:
        if len(buf) + len(s) + 1 <= max_chars:
            buf = (buf + " " + s).strip()
        else:
            if buf:
                chunks.append(buf)
            # Hard-split sentences that exceed the cap on their own
            if len(s) > max_chars:
                for i in range(0, len(s), max_chars):
                    chunks.append(s[i:i + max_chars])
                buf = ""
            else:
                buf = s
    if buf:
        chunks.append(buf)
    return chunks


def _translate_chunk(chunk: str) -> str:
    """Translate a single chunk with retries."""
    last_err = None
    for attempt in range(RETRY):
        try:
            return GoogleTranslator(source="nl", target="en").translate(chunk)
        except Exception as e:
            last_err = e
            time.sleep(1.0 * (attempt + 1))
    # All retries failed — return original with marker so we know
    return f"[TRANSLATION_FAILED: {last_err}] {chunk}"


def translate_text(text) -> Optional[str]:
    """Public API: translate one cell (string or NaN) → English or None."""
    if not _is_translatable(text):
        return text  # pass through unchanged
    s = str(text).strip()
    # Cache lookup
    key = hashlib.sha1(s.encode("utf-8")).hexdigest()
    if key in _cache:
        return _cache[key]
    # Chunk → translate → reassemble
    chunks = _chunk_text(s)
    translated_chunks = []
    for c in chunks:
        translated_chunks.append(_translate_chunk(c))
        time.sleep(DELAY_SEC)
    out = " ".join(translated_chunks)
    _cache[key] = out
    return out


def translate_dataframe(df: pd.DataFrame,
                        columns_to_translate: List[str],
                        progress_every: int = 10) -> pd.DataFrame:
    """
    Translate specified columns in df, adding `<col>_en` siblings.
    Preserves original Dutch columns untouched.
    """
    out = df.copy()
    total_cells = sum(out[c].notna().sum() for c in columns_to_translate)
    done = 0
    t0 = time.time()
    for col in columns_to_translate:
        en_col = f"{col}_en"
        translations = []
        for idx, val in out[col].items():
            translations.append(translate_text(val))
            done += 1
            if done % progress_every == 0:
                elapsed = time.time() - t0
                rate = done / max(elapsed, 1e-6)
                eta = (total_cells - done) / max(rate, 1e-6)
                print(f"  [{done}/{total_cells}] cells | "
                      f"elapsed {elapsed:.0f}s | ETA {eta:.0f}s | "
                      f"cache_size={len(_cache)}")
        out[en_col] = translations
    print(f"  ✅ Done {total_cells} cells in {time.time() - t0:.0f}s "
          f"(cache hits saved time on repeats)")
    return out


def get_cache_stats() -> dict:
    return {"cached_strings": len(_cache)}


## 3. Configure paths and columns

Change `SRC` if your file lives elsewhere. `COLUMNS` lists the Dutch-text columns to translate — based on the schema-fit validation we did before starting.

In [3]:
SRC = '/Users/latafat/Desktop/LSE Data Analytics Program/Employer Project /CSV Folder/kanker_nl_master_table_breakout_v3.csv'
DST = '/Users/latafat/Desktop/LSE Data Analytics Program/Employer Project /Translation /Translated_output/kanker_nl_master_table_breakout_v3_translated.csv'
COLUMNS = ['General Description', 'Prognosis (Vooruitzichten)', 'Symptoms (Symptomen)', 'Causes (Oorzaken)', 'Treatments (Behandeling)']
TRUSTWORTHINESS = 4.9   # from Makhotso's source-selection matrix


## 4. Load the source CSV

In [4]:
import pandas as pd
df = pd.read_csv(SRC)
print(f'Loaded {SRC}: {df.shape[0]} rows × {df.shape[1]} cols')
print('Columns:', list(df.columns))
df.head(2)


Loaded /Users/latafat/Desktop/LSE Data Analytics Program/Employer Project /CSV Folder/kanker_nl_master_table_breakout_v3.csv: 73 rows × 9 cols
Columns: ['Cancer Type', 'Source URL', 'Scrape Date', 'Content Hash', 'General Description', 'Prognosis (Vooruitzichten)', 'Symptoms (Symptomen)', 'Causes (Oorzaken)', 'Treatments (Behandeling)']


,Cancer Type,Source URL,Scrape Date,Content Hash,General Description,Prognosis (Vooruitzichten),Symptoms (Symptomen),Causes (Oorzaken),Treatments (Behandeling)
0,Atypisch fibroxanthoom,https://www.kanker.nl/kankersoorten/atypisch-f...,2026-05-31 14:54:16,1e9d9d820125ef55a3a41c4a2e52dae16c9ea16aac5adb...,Een atypisch fibroxanthoomis een zeldzame soor...,Een atypisch fibroxanthoom is goed te behandel...,Een atypisch fibroxanthoom is een knobbel op j...,De belangrijkste risicofactor voor een atypisc...,Heb je een atypisch fibroxanthoom? Dan krijg j...
1,Anuskanker,https://www.kanker.nl/kankersoorten/anuskanker,2026-05-31 14:54:16,7d98bfe6993858028e2ba2907643560fd902476999d88e...,Anuskanker is kanker die ontstaat in de anus. ...,Veel mensen gaan op zoek naar wat anuskanker b...,Bij anuskanker kan zitten of poepen pijnlijk z...,De oorzaak van anuskanker is meestal een infec...,Er zijn verschillende behandelingen mogelijk b...


## 5. Translate Dutch → English

The pipeline:
1. **Filter** — skip empties, URLs, hashes, dates
2. **Chunk** — split long text on sentence boundaries (≤ 4500 chars per chunk)
3. **Translate** — call Google Translate via deep-translator
4. **Cache** — identical strings translated only once
5. **Reassemble** — chunks joined back into the cell
6. **Side-by-side** — Dutch column preserved; English added as `<col>_en`

In [5]:
translated = translate_dataframe(df, COLUMNS, progress_every=25)
print()
print(f'Output shape: {translated.shape}')
print(f'New columns added: {[c for c in translated.columns if c.endswith("_en")]}')


  [25/344] cells | elapsed 39s | ETA 498s | cache_size=25
  [50/344] cells | elapsed 82s | ETA 482s | cache_size=50
  [75/344] cells | elapsed 123s | ETA 442s | cache_size=75
  [100/344] cells | elapsed 162s | ETA 396s | cache_size=97
  [125/344] cells | elapsed 198s | ETA 348s | cache_size=118
  [150/344] cells | elapsed 235s | ETA 305s | cache_size=141
  [175/344] cells | elapsed 275s | ETA 266s | cache_size=163
  [200/344] cells | elapsed 312s | ETA 225s | cache_size=184
  [225/344] cells | elapsed 346s | ETA 183s | cache_size=206
  [250/344] cells | elapsed 376s | ETA 141s | cache_size=223
  [275/344] cells | elapsed 399s | ETA 100s | cache_size=237
  [300/344] cells | elapsed 440s | ETA 64s | cache_size=260
  [325/344] cells | elapsed 476s | ETA 28s | cache_size=281
  [350/344] cells | elapsed 517s | ETA -9s | cache_size=305
  ✅ Done 344 cells in 539s (cache hits saved time on repeats)

Output shape: (73, 14)
New columns added: ['General Description_en', 'Prognosis (Vooruitzichten

## 6. Spot-check translation quality

In [6]:
for col in COLUMNS:
    en_col = f'{col}_en'
    if en_col not in translated.columns:
        continue
    sample = translated[[col, en_col]].dropna().head(1)
    if sample.empty:
        continue
    print('=' * 60)
    print(f'COLUMN: {col}')
    print(f'NL: {str(sample[col].iloc[0])[:200]}')
    print(f'EN: {str(sample[en_col].iloc[0])[:200]}')
    print()


COLUMN: General Description
NL: Een atypisch fibroxanthoomis een zeldzame soort huidkanker. Deze soort huidkanker groeit snel, maar zaait bijna nooit uit.
EN: Atypical fibroxanthoma is a rare type of skin cancer. This type of skin cancer grows quickly, but almost never spreads.

COLUMN: Prognosis (Vooruitzichten)
NL: Een atypisch fibroxanthoom is goed te behandelen. De kans dat je geneest is heel groot.
EN: Atypical fibroxanthoma is easy to treat. The chance that you will be cured is very high.

COLUMN: Symptoms (Symptomen)
NL: Een atypisch fibroxanthoom is een knobbel op je huid, meestal roze of rood. Soms zit er een wondje op de knobbel. Soms doet de knobbel pijn. Lees hoe een atypisch fibroxanthoom eruitziet .
EN: An atypical fibroxanthoma is a lump on your skin, usually pink or red. Sometimes there is a wound on the lump. Sometimes the lump hurts. Read what an atypical fibroxanthoma looks like.

COLUMN: Causes (Oorzaken)
NL: De belangrijkste risicofactor voor een atypisch fibroxant

## 7. Add trustworthiness column and save

Hard-codes the trustworthiness score (since none of the source files contained it). Saves a UTF-8 CSV with all original columns + new `_en` columns.

In [7]:
translated['trustworthiness'] = TRUSTWORTHINESS
translated.to_csv(DST, index=False, encoding='utf-8')
print(f'💾 Saved {DST}')
print(f'   Rows: {translated.shape[0]}')
print(f'   Cols: {translated.shape[1]}')
print(f'   Cache hits during this run: {get_cache_stats()}')


💾 Saved /Users/latafat/Desktop/LSE Data Analytics Program/Employer Project /Translation /Translated_output/kanker_nl_master_table_breakout_v3_translated.csv
   Rows: 73
   Cols: 15
   Cache hits during this run: {'cached_strings': 319}


## ✅ Done

Hand the resulting CSV to Quadry for inclusion in the consolidated `csn_knowledge` table.  
The Dutch columns are preserved alongside the English ones so any reviewer can spot-check translations.